# 第12讲：数据清洗与准备

## 课程简介

在真实的数据分析项目中，数据往往不会一开始就“干净整齐”。  
在分析、建模和可视化之前，我们通常需要先进行数据准备，包括缺失值处理、重复值处理、异常值处理、类型转换和字符串清洗等。

本讲将围绕 Pandas 中常用的数据清洗工具展开。学习完成后，你应该能够：

- 识别、删除和填充缺失数据
- 发现并移除重复数据
- 使用映射、替换和重命名完成数据转换
- 使用 `cut()` 和 `qcut()` 对连续变量离散化
- 检测和处理异常值
- 随机重排与抽样
- 将分类变量转换为哑变量
- 使用扩展数据类型和字符串向量化方法清洗文本数据


## 1 导入库

In [ ]:
import numpy as np
import pandas as pd
import re

np.random.seed(12)

pd.options.display.max_rows = 25
pd.options.display.max_columns = 20
pd.options.display.max_colwidth = 82
np.set_printoptions(precision=4, suppress=True)

print("Pandas 版本：", pd.__version__)
print("NumPy 版本：", np.__version__)


### <font color='limegreen'><b>技巧与提示</b></font>

数据清洗的目标不是“机械地修改数据”，而是让数据更适合后续分析。  
在执行删除、填充、替换等操作之前，最好先明确：

- 为什么这些值是缺失或异常的？
- 删除数据会不会丢失重要信息？
- 填充值是否符合业务含义？
- 转换后的数据类型是否适合后续分析？


## 2 处理缺失数据

在数据分析中，缺失数据非常常见。  
Pandas 的目标之一，就是让缺失数据的检测、过滤和填充尽可能简单。

对于数值数据，Pandas 通常使用 `NaN` 表示缺失值；对于较新的扩展数据类型，也可以使用 `pd.NA` 表示缺失值。


In [ ]:
float_data = pd.Series([1.2, -3.5, np.nan, 0])
float_data

In [ ]:
float_data.isna()

In [ ]:
string_data = pd.Series(["aardvark", np.nan, None, "avocado"])
string_data
string_data.isna()

### 缺失值处理常用方法

| 方法 | 说明 |
|---|---|
| `isna()` | 判断是否为缺失值 |
| `notna()` | 判断是否不是缺失值 |
| `dropna()` | 删除缺失值所在的行或列 |
| `fillna()` | 用指定值或规则填充缺失值 |
| `ffill()` | 使用前一个有效值向前填充 |
| `bfill()` | 使用后一个有效值向后填充 |

这些方法是数据清洗中最常用的工具。


### 2.1 滤除缺失数据

过滤缺失数据的方法有很多。  
对于 Series，`dropna()` 会返回仅包含非缺失值的新 Series。


In [ ]:
data = pd.Series([1, np.nan, 3.5, np.nan, 7])
data

In [ ]:
data.dropna()

In [ ]:
# 这等价于：
data[data.notna()]

对于 DataFrame，`dropna()` 默认会删除任何包含缺失值的行。

In [ ]:
data = pd.DataFrame([[1., 6.5, 3.], [1., np.nan, np.nan],
                     [np.nan, np.nan, np.nan], [np.nan, 6.5, 3.]])
data

In [ ]:
data.dropna()

如果只想删除所有值都缺失的行，可以传入 `how="all"`。

In [ ]:
data.dropna(how="all")

删除列时，可以使用 `axis="columns"`。

In [ ]:
data[4] = np.nan
data

In [ ]:
data.dropna(axis="columns", how="all")

有时不希望所有含缺失值的行都被删除，而是希望至少保留有一定数量有效值的行。  
这时可以使用 `thresh` 参数。


In [ ]:
df = pd.DataFrame(np.random.standard_normal((7, 3)))
df.iloc[:4, 1] = np.nan
df.iloc[:2, 2] = np.nan
df

In [ ]:
df.dropna()

In [ ]:
df.dropna(thresh=2)

### 2.2 填充缺失数据

如果不希望直接删除缺失值，可以用 `fillna()` 或前后填充方法补全数据。  
这在时间序列、调查问卷、实验记录等场景中很常见。


In [ ]:
df.fillna(0)

也可以传入字典，对不同列使用不同的填充值。

In [ ]:
df.fillna({1: 0.5, 2: 0})

对于有顺序的数据，可以使用前向填充或后向填充。  
在现代 Pandas 中，推荐直接使用 `.ffill()` 或 `.bfill()`。


In [ ]:
df = pd.DataFrame(np.random.standard_normal((6, 3)))
df.iloc[2:, 1] = np.nan
df.iloc[4:, 2] = np.nan
df

In [ ]:
df.ffill()


In [ ]:
df.ffill(limit=2)


还可以使用统计量进行填充，例如均值、中位数或众数。

In [ ]:
data = pd.Series([1., np.nan, 3.5, np.nan, 7])
data

In [ ]:
data.fillna(data.mean())

### <font color='darkorange'><b>动手练习 1</b></font>

#### 题目
下方是一个包含缺失值的 DataFrame，请完成：

1. 查看每一列缺失值数量
2. 删除全为缺失值的列
3. 用每列均值填充数值列缺失值

#### 你的答案
请在下方代码单元中完成练习。


In [ ]:
practice = pd.DataFrame({
    "A": [1.0, np.nan, 3.0],
    "B": [np.nan, np.nan, np.nan],
    "C": [10.0, 20.0, np.nan]
})

# Write your code here



#### 参考答案

<details>
<summary>点击查看示例代码</summary>

```python
practice = pd.DataFrame({
    "A": [1.0, np.nan, 3.0],
    "B": [np.nan, np.nan, np.nan],
    "C": [10.0, 20.0, np.nan]
})

print(practice.isna().sum())

practice = practice.dropna(axis="columns", how="all")
practice = practice.fillna(practice.mean(numeric_only=True))
practice
```

</details>


## 3 数据转换

除了处理缺失值，数据清洗还包括过滤、去重、替换、重命名、离散化、异常值处理等转换操作。

### 3.1 移除重复数据

DataFrame 中出现重复行很常见，例如重复导入、重复录入或多个系统合并后的重复记录。


In [ ]:
data = pd.DataFrame({"k1": ["one", "two"] * 3 + ["two"],
                     "k2": [1, 1, 2, 3, 3, 4, 4]})
data

`duplicated()` 会返回一个布尔 Series，表示每一行是否是重复行。  
默认情况下，第一次出现的行不算重复，后续重复出现的行会标记为 `True`。


In [ ]:
data.duplicated()

`drop_duplicates()` 会返回删除重复行之后的新 DataFrame。

In [ ]:
data.drop_duplicates()

这两个方法默认会比较所有列，也可以通过 `subset` 指定只根据部分列判断重复。

In [ ]:
data["v1"] = range(7)
data

In [ ]:
data.drop_duplicates(subset=["k1"])

默认保留第一次出现的记录；如果传入 `keep="last"`，则保留最后一次出现的记录。

In [ ]:
data.drop_duplicates(["k1", "k2"], keep="last")

### 3.2 利用函数或映射进行数据转换

对于许多数据集，我们经常需要根据某一列的取值生成新的信息。  
例如，根据食物名称映射出来源动物。


In [ ]:
data = pd.DataFrame({"food": ["bacon", "pulled pork", "bacon",
                              "pastrami", "corned beef", "bacon",
                              "pastrami", "honey ham", "nova lox"],
                     "ounces": [4, 3, 12, 6, 7.5, 8, 3, 5, 6]})
data

假设你想要添加一列表示该肉类食物来源的动物类型。我们先编写一个不同肉类到动物的映射：

In [ ]:
meat_to_animal = {
  "bacon": "pig",
  "pulled pork": "pig",
  "pastrami": "cow",
  "corned beef": "cow",
  "honey ham": "pig",
  "nova lox": "salmon"
}

Series 的 `map()` 可以接收字典或函数，非常适合做元素级转换。

In [ ]:
data["animal"] = data["food"].map(meat_to_animal)
data

也可以将映射逻辑封装成函数，然后传给 `map()`。

In [ ]:
def get_animal(x):
    return meat_to_animal[x]
    
data["food"].map(get_animal)
data

使用 `map()` 是进行元素级转换和数据清理的常用方式。

### 3.3 替换值

`replace()` 可以将某些值替换为其他值。  
这常用于把特殊标记值（例如 `-999`）替换成 Pandas 能识别的缺失值。


In [ ]:
data = pd.Series([1., -999., 2., -999., -1000., 3.])
data

例如，`-999` 可能表示缺失数据，可以将其替换为 `np.nan`。

In [ ]:
data.replace(-999, np.nan)

如果要一次替换多个值，可以传入一个待替换值列表。

In [ ]:
data.replace([-999, -1000], np.nan)

如果不同值要替换成不同结果，可以分别传入两个列表。

In [ ]:
data.replace([-999, -1000], [np.nan, 0])

也可以用字典表示替换关系。

In [ ]:
data.replace({-999: np.nan, -1000: 0})

### <font color='limegreen'><b>技巧与提示</b></font>

`Series.replace()` 和 `Series.str.replace()` 不同：

- `replace()` 用于替换 Series 中的值
- `str.replace()` 用于对字符串内容做元素级替换


### <font color='darkorange'><b>动手练习 2</b></font>

#### 题目
给定一个订单表，请完成：

1. 删除重复订单
2. 将状态列中的 `"unknown"` 替换为缺失值
3. 根据状态映射出中文状态名称

#### 你的答案
请在下方代码单元中完成练习。


In [ ]:
orders = pd.DataFrame({
    "order_id": [1, 1, 2, 3, 4],
    "status": ["paid", "paid", "pending", "unknown", "paid"]
})

# Write your code here



#### 参考答案

<details>
<summary>点击查看示例代码</summary>

```python
orders = pd.DataFrame({
    "order_id": [1, 1, 2, 3, 4],
    "status": ["paid", "paid", "pending", "unknown", "paid"]
})

orders = orders.drop_duplicates()
orders["status"] = orders["status"].replace("unknown", np.nan)

status_map = {
    "paid": "已支付",
    "pending": "待处理"
}

orders["status_cn"] = orders["status"].map(status_map)
orders
```

</details>


### 3.4 重命名轴索引

行索引和列名也可以通过函数或字典进行转换。  
如果不想直接修改原对象，推荐使用 `rename()` 返回一个新对象。


In [ ]:
data = pd.DataFrame(np.arange(12).reshape((3, 4)),
                    index=["Ohio", "Colorado", "New York"],
                    columns=["one", "two", "three", "four"])
data

Index 对象也有 `map()` 方法，可以对每个标签应用转换函数。

In [ ]:
def transform(x):
    return x[:4].upper()

data.index.map(transform)

可以将转换后的结果重新赋给 `index`，从而修改 DataFrame 的行索引。

In [ ]:
data.index = data.index.map(transform)
data

如果希望创建一个转换后的新对象，而不是直接修改原对象，可以使用 `rename()`。

In [ ]:
data.rename(index=str.title, columns=str.upper)

`rename()` 也可以接收字典，只重命名指定的行或列标签。

In [ ]:
data.rename(index={"OHIO": "INDIANA"},
            columns={"three": "peekaboo"})

### 3.5 离散化和面元划分

为了便于分析，连续数据常常需要被划分为若干区间，也叫“分箱”或“面元划分”。  
例如，可以把年龄划分为青年、中年、老年等组别。


In [ ]:
ages = [20, 22, 25, 27, 21, 23, 37, 31, 61, 45, 41, 32]

可以使用 `pd.cut()` 按指定边界划分区间。

In [ ]:
bins = [18, 25, 35, 60, 100]
age_categories = pd.cut(ages, bins)
age_categories

`pd.cut()` 返回的是一个 Categorical 对象，其中包含每个数据点所属的区间。

In [ ]:
age_categories.codes

In [ ]:
age_categories.categories

In [ ]:
age_categories.categories[0]

In [ ]:
age_categories.value_counts()


区间符号中，圆括号表示开区间，方括号表示闭区间。  
可以通过 `right=False` 改变哪一侧为闭区间。


In [ ]:
pd.cut(ages, bins, right=False)

也可以通过 `labels` 参数设置更易读的分组名称。

In [ ]:
group_names = ["Youth", "YoungAdult", "MiddleAged", "Senior"]
pd.cut(ages, bins, labels=group_names)

如果传入的是分箱数量，而不是具体边界，`pd.cut()` 会根据数据最小值和最大值自动计算等宽区间。

In [ ]:
data = np.random.uniform(size=20)
data

In [ ]:
pd.cut(data, 4, precision=2)

`pd.qcut()` 会根据分位数划分区间，通常能让每个区间中的样本数量大致相同。

In [ ]:
data = np.random.standard_normal(1000)
quartiles = pd.qcut(data, 4, precision=2)
quartiles

In [ ]:
quartiles.value_counts()


也可以向 `qcut()` 传入自定义分位点。

In [ ]:
pd.qcut(data, [0, 0.1, 0.5, 0.9, 1.]).value_counts()

`cut()` 和 `qcut()` 在分组分析中非常有用，后续学习聚合和分组时还会再次用到。

### <font color='cornflowerblue'><b>思考题 1</b></font>

`cut()` 和 `qcut()` 有什么区别？

#### 参考答案

<details>
<summary>点击查看解释</summary>

- `cut()` 根据给定边界或等宽区间切分数据，更关注“区间范围”
- `qcut()` 根据分位数切分数据，更关注“每组样本数量接近”

如果希望年龄按固定范围分组，常用 `cut()`；如果希望每组样本数量大致相同，常用 `qcut()`。

</details>


### 3.6 检测和过滤异常值

异常值是指明显偏离大多数样本的值。  
处理异常值的方法包括删除、截断、替换或单独标记。


In [ ]:
data = pd.DataFrame(np.random.randn(1000, 4))
data.describe()

下面选出第 2 列中绝对值大于 3 的数据。

In [ ]:
col = data[2]
col[col.abs() > 3]

如果想选出任意列中存在极端值的行，可以对布尔 DataFrame 使用 `.any(axis="columns")`。

In [ ]:
data[(data.abs() > 3).any(axis="columns")]

下面将所有值限制在 `[-3, 3]` 区间内。

In [ ]:
data = data.clip(lower=-3, upper=3)
data.describe()


`np.sign()` 可以返回数据的符号：正数为 1，负数为 -1，0 为 0。

In [ ]:
np.sign(data).head()

### 3.7 排列和随机采样

随机重排和抽样常用于构造训练集、测试集，或者进行模拟实验。  
Pandas 提供了 `take()`、`iloc[]` 和 `sample()` 等方法。


In [ ]:
df = pd.DataFrame(np.arange(5 * 7).reshape((5, 7)))
df

In [ ]:
sampler = np.random.permutation(5)
sampler

可以在基于 `iloc` 的索引操作中使用随机排列结果。

In [ ]:
df.take(sampler)

In [ ]:
df.iloc[sampler]

In [ ]:
column_sampler = np.random.permutation(7)
column_sampler
df.take(column_sampler, axis="columns")

如果只想随机选取一部分样本，可以使用 `sample()`。

In [ ]:
df.sample(n=3)

如果允许重复抽样，可以设置 `replace=True`。

In [ ]:
choices = pd.Series([5, 7, -1, 6, 4])
choices.sample(n=10, replace=True)

### 3.8 计算指标变量/哑变量

在统计建模和机器学习中，分类变量经常需要转换成哑变量。  
Pandas 的 `get_dummies()` 可以完成这个操作。


In [ ]:
df = pd.DataFrame({"key": ["b", "b", "a", "c", "a", "b"],
                   "data1": range(6)})
df

In [ ]:
pd.get_dummies(df["key"], dtype=float)

可以通过 `prefix` 参数给哑变量列名添加前缀，方便与原数据合并。

In [ ]:
dummies = pd.get_dummies(df["key"], prefix="key", dtype=float)
df_with_dummy = df[["data1"]].join(dummies)
df_with_dummy

### <font color='darkorange'><b>动手练习 3</b></font>

#### 题目
请创建一个包含 `gender` 和 `city` 两个分类变量的 DataFrame，并使用 `pd.get_dummies()` 生成哑变量。

#### 你的答案
请在下方代码单元中完成练习。


In [ ]:
# Write your code here



#### 参考答案

<details>
<summary>点击查看示例代码</summary>

```python
people = pd.DataFrame({
    "gender": ["F", "M", "F", "M"],
    "city": ["Beijing", "Shanghai", "Beijing", "Shenzhen"]
})

pd.get_dummies(people, dtype=int)
```

</details>


## 4 拓展数据类型

NumPy 主要面向数值数组。Pandas 在此基础上扩展了更多适合数据分析的数据类型，例如：

- 可缺失整数类型：`Int64`
- 字符串类型：`string`
- 可缺失布尔类型：`boolean`
- 分类类型：`category`

这些类型可以更好地表示真实数据中的缺失值和混合类型。


In [ ]:
s = pd.Series([1, 2, 3, None])
s

In [ ]:
s.dtype

默认情况下，如果整数 Series 中出现缺失值，Pandas 可能会将其转换为浮点类型。  
可以使用可缺失整数类型 `Int64` 避免这个问题。


In [ ]:
s = pd.Series([1, 2, 3, None], dtype=pd.Int64Dtype())
s

In [ ]:
s.isna()

In [ ]:
s.dtype

这里的 `<NA>` 表示 Pandas 扩展类型中的缺失值。

In [ ]:
s[3]

In [ ]:
s[3] is pd.NA

也可以直接使用字符串 `"Int64"` 指定可缺失整数类型。  
注意这里的 `I` 必须大写。


In [ ]:
s = pd.Series([1, 2, 3, None], dtype="Int64")

Pandas 也提供了专门的字符串扩展类型。

In [ ]:
s = pd.Series(['one', 'two', None, 'three'], dtype=pd.StringDtype())
s

扩展类型可以通过 `astype()` 进行转换，这在数据清洗中非常常见。

In [ ]:
df = pd.DataFrame({"A": [1, 2, None, 4],
                   "B": ["one", "two", "three", None],
                   "C": [False, None, False, True]})
df

In [ ]:
df["A"] = df["A"].astype("Int64")
df["B"] = df["B"].astype("string")
df["C"] = df["C"].astype("boolean")
df

### <font color='limegreen'><b>技巧与提示</b></font>

在数据清洗中，选择合适的数据类型很重要。  
例如：

- 有缺失值的整数列可以使用 `"Int64"`
- 字符串列可以使用 `"string"`
- 布尔列可以使用 `"boolean"`
- 类别较少的字符串列可以考虑使用 `"category"`

这样可以让数据含义更明确，也可能减少内存占用。


## 5 字符串操作

真实数据中经常包含姓名、地址、邮箱、类别标签等字符串数据。  
Pandas 提供了丰富的字符串处理能力，可以对整列字符串进行向量化操作。

### 5.1 字符串对象方法

Python 字符串本身就有许多常用方法，例如 `split()`、`strip()`、`join()`、`find()`、`replace()` 等。


In [ ]:
val = "a,b,  guido"
val.split(",")

`split()` 通常会和 `strip()` 一起使用，用于拆分并清理空白字符。

In [ ]:
pieces = [x.strip() for x in val.split(",")]
pieces

可以用加号拼接字符串，但更推荐使用 `join()`。

In [ ]:
first, second, third = pieces
first + "::" + second + "::" + third

`join()` 通常更简洁，也更符合 Python 风格。

In [ ]:
"::".join(pieces)

检测子串是否存在，最推荐使用 `in` 关键字。  
如果需要位置，可以使用 `find()` 或 `index()`。


In [ ]:
"guido" in val

In [ ]:
val.index(",")

In [ ]:
val.find(":") # no :

`find()` 和 `index()` 的区别是：找不到时，`find()` 返回 `-1`，而 `index()` 会抛出异常。

In [ ]:
try:
    val.index(":")
except ValueError as err:
    print("index() 找不到子串时会抛出异常。")
    print("错误信息：", err)


与此相关，count可以返回指定子串的出现次数：

In [ ]:
val.count(",")

`replace()` 可以替换子串，也可以通过替换为空字符串来删除子串。

In [ ]:
val.replace(",", "::")

In [ ]:
val.replace(",", "")

### Python 常用字符串方法

| 方法 | 说明 |
|---|---|
| `split()` | 按分隔符拆分字符串 |
| `strip()` | 去除首尾空白字符 |
| `join()` | 将多个字符串连接起来 |
| `find()` | 查找子串位置，找不到返回 -1 |
| `index()` | 查找子串位置，找不到抛出异常 |
| `count()` | 统计子串出现次数 |
| `replace()` | 替换子串 |
| `lower()` / `upper()` | 大小写转换 |


### 5.2 正则表达式

正则表达式提供了一种灵活的文本匹配工具，常用于搜索、提取、替换和拆分字符串。  
Python 内置的 `re` 模块可以处理正则表达式。


In [ ]:
import re
text = "foo    bar\t baz  \tqux"
re.split(r"\s+", text)

如果同一个正则表达式会反复使用，可以先用 `re.compile()` 编译成正则对象。

In [ ]:
regex = re.compile(r"\s+")
regex.split(text)

如果只希望得到所有匹配内容，可以使用 `findall()`。

In [ ]:
regex.findall(text)

### <font color='limegreen'><b>技巧与提示</b></font>

写正则表达式时，推荐使用原始字符串，例如 `r"\s+"`。  
这样可以避免反斜杠被 Python 字符串转义规则提前处理。

如果同一个正则表达式要使用很多次，可以用 `re.compile()` 提前编译，提高可读性和效率。


In [ ]:
text = """Dave dave@google.com
Steve steve@gmail.com
Rob rob@gmail.com
Ryan ryan@yahoo.com"""
pattern = r"[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,4}"

# re.IGNORECASE makes the regex case insensitive
regex = re.compile(pattern, flags=re.IGNORECASE)

这个正则表达式可以匹配大多数常见邮箱地址，但并不能覆盖所有合法邮箱格式。  
在教学和数据清洗中，它足以演示正则表达式的基本用法。


对文本使用 `findall()` 可以得到所有匹配的电子邮件地址。

In [ ]:
regex.findall(text)

`search()` 返回文本中第一个匹配项，结果是一个匹配对象。

In [ ]:
m = regex.search(text)
m

In [ ]:
text[m.start():m.end()]

`match()` 只从字符串开头开始匹配。由于这里的文本开头不是邮箱地址，所以返回 `None`。

In [ ]:
print(regex.match(text))

`sub()` 可以将匹配内容替换为指定字符串。

In [ ]:
print(regex.sub("REDACTED", text))

如果不仅想找到邮箱地址，还想提取用户名、域名和后缀，可以用圆括号创建分组。

In [ ]:
pattern = r"([A-Z0-9._%+-]+)@([A-Z0-9.-]+)\.([A-Z]{2,4})"
regex = re.compile(pattern, flags=re.IGNORECASE)

带分组的匹配对象可以使用 `.groups()` 返回各分组结果。

In [ ]:
m = regex.match("wesm@bright.net")
m.groups()

对带分组的正则表达式使用 `findall()`，会得到元组列表。

In [ ]:
regex.findall(text)

在 `sub()` 中，可以使用 `\1`、`\2` 等引用正则表达式中的分组。

In [ ]:
print(regex.sub(r"Username: \1, Domain: \2, Suffix: \3", text))

### 正则表达式常用方法

| 方法 | 说明 |
|---|---|
| `re.split()` | 按模式拆分字符串 |
| `regex.findall()` | 返回所有匹配项 |
| `regex.search()` | 返回第一个匹配项 |
| `regex.match()` | 只从字符串开头匹配 |
| `regex.sub()` | 替换匹配内容 |
| `match.groups()` | 获取分组匹配结果 |


### 5.3 Pandas 的向量化字符串函数

在清洗文本列时，使用普通 Python 字符串方法可能会遇到缺失值问题。  
Pandas 的 `.str` 访问器可以对整列字符串进行向量化处理，并自动跳过缺失值。


In [ ]:
data = {"Dave": "dave@google.com", "Steve": "steve@gmail.com",
        "Rob": "rob@gmail.com", "Wes": np.nan}
data = pd.Series(data)
data

In [ ]:
data.isna()

如果直接用 `map()` 调用普通字符串方法，遇到缺失值可能会报错。  
使用 `.str` 方法可以更安全地处理包含缺失值的字符串列。


In [ ]:
data.str.contains("gmail")

`.str` 方法也支持正则表达式，并且可以传入 `flags` 等参数。

In [ ]:
pattern = r"([A-Z0-9._%+-]+)@([A-Z0-9.-]+)\.([A-Z]{2,4})"
data.str.findall(pattern, flags=re.IGNORECASE)

对于提取出的匹配结果，可以用 `.str.get()` 或索引方式获取其中的某一部分。

In [ ]:
matches = data.str.findall(pattern, flags=re.IGNORECASE).str[0]
matches

In [ ]:
matches.str.get(1)

In [ ]:
data.str[:5]

`str.extract()` 会把正则表达式中的分组提取为 DataFrame。

In [ ]:
data.str.extract(pattern, flags=re.IGNORECASE)

### <font color='darkorange'><b>动手练习 4</b></font>

#### 题目
给定一个包含邮箱地址的 Series，请完成：

1. 判断每个邮箱是否来自 `gmail`
2. 提取用户名
3. 提取域名

#### 你的答案
请在下方代码单元中完成练习。


In [ ]:
emails = pd.Series(["alice@gmail.com", "bob@qq.com", np.nan, "cathy@gmail.com"])

# Write your code here



#### 参考答案

<details>
<summary>点击查看示例代码</summary>

```python
emails = pd.Series(["alice@gmail.com", "bob@qq.com", np.nan, "cathy@gmail.com"])

print(emails.str.contains("gmail", na=False))

pattern = r"([A-Z0-9._%+-]+)@([A-Z0-9.-]+)"
parts = emails.str.extract(pattern, flags=re.IGNORECASE)
parts.columns = ["username", "domain"]
parts
```

</details>


### Pandas 常用字符串方法

| 方法 | 说明 |
|---|---|
| `str.contains()` | 判断是否包含某模式 |
| `str.findall()` | 找出所有匹配项 |
| `str.extract()` | 提取正则分组 |
| `str.get()` | 获取列表或元组中的指定位置 |
| `str.lower()` / `str.upper()` | 大小写转换 |
| `str.strip()` | 去除首尾空白 |
| `str.replace()` | 字符串替换 |
| `str.len()` | 计算字符串长度 |
| `str.slice()` | 字符串切片 |


## 课堂小结

在本讲中，我们学习了 Pandas 中常用的数据清洗和准备方法。

| 知识点 | 主要内容 |
|---|---|
| **缺失值检测** | `isna()`、`notna()` |
| **缺失值删除** | `dropna()` |
| **缺失值填充** | `fillna()`、`ffill()`、`bfill()` |
| **重复值处理** | `duplicated()`、`drop_duplicates()` |
| **映射转换** | `map()` |
| **值替换** | `replace()` |
| **重命名标签** | `rename()` |
| **离散化** | `cut()`、`qcut()` |
| **异常值处理** | 条件过滤、`clip()` |
| **随机抽样** | `sample()` |
| **哑变量** | `get_dummies()` |
| **扩展数据类型** | `Int64`、`string`、`boolean` |
| **字符串清洗** | `.str` 向量化字符串方法 |
| **正则表达式** | `re` 模块、`str.extract()` |


### <font color='cornflowerblue'><b>思考题 2</b></font>

为什么数据清洗通常比建模更耗时？

#### 参考答案

<details>
<summary>点击查看解释</summary>

真实数据往往存在缺失值、重复值、异常值、格式不统一、编码不一致、字段含义不清等问题。  
如果不先处理这些问题，后续统计分析或模型训练的结果可能不可靠。

因此，数据清洗不是简单的前置步骤，而是决定分析质量的重要环节。

</details>


<div class="alert alert-success">

**进一步学习资源**

- Pandas 官方文档：https://pandas.pydata.org/docs/
- Pandas 官方教程：https://pandas.pydata.org/docs/getting_started/index.html
- Pandas 速查表（Cheat Sheet）：https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf

</div>

---

*本节课到此结束，感谢大家的学习！如有疑问，请随时提问。*